In [1]:
import pandas as pd
import numpy as np
import re
import os
import requests
from io import StringIO
import time
from openpyxl import load_workbook
import shutil
import subprocess
import tempfile

# ============================================================
# (0) PATHS
# ============================================================

# Monthly Umrah files
PATH_1445_UMRAH_MONTHLY = "/content/Number of Umrah permits by user type.xlsx"
PATH_1446_UMRAH_MONTHLY = "/content/Umrah permits for the year 1446 by month and type.xlsx"

# Ramadan daily files
PATH_1445_RAMADAN_DAILY = "/content/Number of Umrah permits in Ramadan.xlsx"
PATH_1446_RAMADAN_DAILY = "/content/Umrah permits by day for the month of Ramadan 1446.xlsx"

# Hajj files
PATH_1445_PILGRIMS_DAILY = "/content/Number of pilgrims from abroad by gender per day.xlsx"
PATH_1445_DOMESTIC = "/content/Number of domestic pilgrims in 1445 by age.xlsx"
PATH_1446_PILGRIMS_ENTER = "/content/Number of pilgrims entering by age group for 1446.xlsx"
PATH_1446_DOMESTIC = "/content/Number of domestic pilgrims by age group for 1446.xlsx"

# Weather file
CLIMATE_PATH = "/content/Average Maximum Temperature, By Month and Meteorological   Station2001 A.D. ( In Centigrade Degrees )_2001.xls"

# New output file
OUT_FILE = "/content/Umrah_Crowd_Forecasting_Dataset_Updated.xlsx"


def resolve_path(p):
    if os.path.exists(p):
        return p

    alt = p.replace("/content/", "/mnt/data/")
    if os.path.exists(alt):
        return alt

    return p


PATH_1445_UMRAH_MONTHLY = resolve_path(PATH_1445_UMRAH_MONTHLY)
PATH_1446_UMRAH_MONTHLY = resolve_path(PATH_1446_UMRAH_MONTHLY)

PATH_1445_RAMADAN_DAILY = resolve_path(PATH_1445_RAMADAN_DAILY)
PATH_1446_RAMADAN_DAILY = resolve_path(PATH_1446_RAMADAN_DAILY)

PATH_1445_PILGRIMS_DAILY = resolve_path(PATH_1445_PILGRIMS_DAILY)
PATH_1445_DOMESTIC = resolve_path(PATH_1445_DOMESTIC)

PATH_1446_PILGRIMS_ENTER = resolve_path(PATH_1446_PILGRIMS_ENTER)
PATH_1446_DOMESTIC = resolve_path(PATH_1446_DOMESTIC)

CLIMATE_PATH = resolve_path(CLIMATE_PATH)
OUT_FILE = resolve_path(OUT_FILE)


# ============================================================
# (1) FINAL COLUMNS
# ============================================================

REQUIRED_COLS = [
    "Hijri_Year",
    "Hijri_Date",
    "Hijri_Month",
    "Gregorian_Date",
    "Weekday_AR",
    "AvgTemp_C",
    "المعتمرين",
    "Hajj",
    "Tawaf_Ifadah",
    "Crowding_Level",
    "Crowding_Level_Num"
]


# ============================================================
# (2) HELPERS
# ============================================================

CANON_MONTHS = {
    1: "محرم",
    2: "صفر",
    3: "ربيع الأول",
    4: "ربيع الآخر",
    5: "جماد الأول",
    6: "جماد الآخر",
    7: "رجب",
    8: "شعبان",
    9: "رمضان",
    10: "شوال",
    11: "ذو القعدة",
    12: "ذو الحجة"
}

weekday_ar = {
    "Sunday": "الأحد",
    "Monday": "الاثنين",
    "Tuesday": "الثلاثاء",
    "Wednesday": "الأربعاء",
    "Thursday": "الخميس",
    "Friday": "الجمعة",
    "Saturday": "السبت"
}


def clean_cols(df):
    df.columns = [str(c).strip() for c in df.columns]
    return df


def norm_ar(s):
    if pd.isna(s):
        return ""

    s = str(s).strip()
    s = re.sub(r"[ـًٌٍَُِّْ]", "", s)

    s = s.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    s = s.replace("ى", "ي").replace("ة", "ه")

    s = " ".join(s.split())
    return s


def month_no_from_any(x):
    if pd.isna(x):
        return np.nan

    raw = str(x).strip()
    s = norm_ar(x)

    if re.fullmatch(r"\d{1,2}", s):
        m = int(s)
        return m if 1 <= m <= 12 else np.nan

    if "محرم" in s:
        return 1
    if "صفر" in s:
        return 2
    if "ربيع" in s and ("اول" in s or "الاول" in s):
        return 3
    if "ربيع" in s and ("اخر" in s or "الاخر" in s or "ثاني" in s or "الثاني" in s):
        return 4

    if ("جماد" in s or "جمادى" in raw) and ("اول" in s or "الاول" in s or "اولي" in s):
        return 5
    if ("جماد" in s or "جمادى" in raw) and ("اخر" in s or "الاخر" in s or "ثاني" in s or "الثاني" in s):
        return 6

    if "رجب" in s:
        return 7
    if "شعبان" in s:
        return 8
    if "رمضان" in s:
        return 9
    if "شوال" in s:
        return 10
    if "ذو" in s and "قعد" in s:
        return 11
    if "ذو" in s and "حج" in s:
        return 12

    return np.nan


def to_float(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()
    s = s.replace("٫", ".")
    s = s.replace(",", "")
    s = s.replace("…", "")
    s = s.replace("-", "")

    trans = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
    s = s.translate(trans)

    try:
        return float(s)
    except:
        return np.nan


def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def HK(d, m, y):
    return f"{int(d):02d}-{int(m):02d}-{int(y)}"


def parse_hijri_ymd_yyyy_mm_dd(series):
    s = series.astype(str).str.strip().str.replace("/", "-", regex=False)
    parts = s.str.split("-", expand=True)

    if parts.shape[1] >= 3:
        y = pd.to_numeric(parts[0], errors="coerce")
        m = pd.to_numeric(parts[1], errors="coerce")
        d = pd.to_numeric(parts[2], errors="coerce")
        return y, m, d

    return (
        pd.Series([np.nan] * len(series)),
        pd.Series([np.nan] * len(series)),
        pd.Series([np.nan] * len(series))
    )


# ============================================================
# (3) HIJRI CALENDAR
# ============================================================

MONTH_SLUG = {
    1: "Muharam",
    2: "Safar",
    3: "Rabiulawal",
    4: "Rabiulakhir",
    5: "Jamadilawal",
    6: "Jamadilakhir",
    7: "Rejab",
    8: "Syaaban",
    9: "Ramadan",
    10: "Syawal",
    11: "Zulkaedah",
    12: "Zulhijah",
}


def fetch_tables_with_headers(url, retries=3, sleep_s=1.2):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9,ar;q=0.8"
    }

    last_err = None

    for _ in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            r.raise_for_status()
            return pd.read_html(StringIO(r.text))
        except Exception as e:
            last_err = e
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed to fetch {url}. Last error: {last_err}")


def fetch_hijri_calendar_year(year, cache_csv=True):
    cache_name = f"hijri_calendar_{year}.csv"

    if cache_csv and os.path.exists(cache_name):
        cal = pd.read_csv(cache_name)
        cal["greg_dt"] = pd.to_datetime(cal["greg_dt"], errors="coerce").dt.date
        return cal

    rows = []

    for m in range(1, 13):
        url = f"https://hijri-calendar.com/en/{MONTH_SLUG[m]}/{year}"
        tables = fetch_tables_with_headers(url)

        t = tables[0].copy()
        t.columns = [str(c).strip() for c in t.columns]

        col_day = pick_col(t, ["Day", "Today", "Weekday"])
        col_hij = pick_col(t, ["Hijri", "Hijri Date", "HijriDate"])
        col_gre = pick_col(t, ["Gregorian", "Gregorian Date", "GregorianDate"])

        if col_day is None or col_hij is None or col_gre is None:
            col_day, col_hij, col_gre = t.columns[:3]

        t = t[[col_day, col_hij, col_gre]].rename(
            columns={
                col_day: "Day_EN",
                col_hij: "Hijri",
                col_gre: "Gregorian"
            }
        )

        t["Hijri"] = t["Hijri"].astype(str).str.strip()
        t = t[t["Hijri"].str.startswith(f"{year}/")].copy()

        parts = t["Hijri"].str.split("/", expand=True)

        t["y"] = pd.to_numeric(parts[0], errors="coerce")
        t["m"] = pd.to_numeric(parts[1], errors="coerce")
        t["d"] = pd.to_numeric(parts[2], errors="coerce")

        t["greg_dt"] = pd.to_datetime(t["Gregorian"], errors="coerce").dt.date

        t = t.dropna(subset=["y", "m", "d", "greg_dt"]).copy()

        t["y"] = t["y"].astype(int)
        t["m"] = t["m"].astype(int)
        t["d"] = t["d"].astype(int)

        t["Hijri_Date"] = t.apply(lambda r: HK(r["d"], r["m"], r["y"]), axis=1)
        t["Hijri_Month_Name"] = t["m"].map(CANON_MONTHS)
        t["Weekday_AR"] = t["Day_EN"].map(weekday_ar).fillna("")

        rows.append(
            t[
                [
                    "y",
                    "m",
                    "d",
                    "Hijri_Date",
                    "Hijri_Month_Name",
                    "greg_dt",
                    "Day_EN",
                    "Weekday_AR"
                ]
            ]
        )

        time.sleep(0.35)

    cal = pd.concat(rows, ignore_index=True)
    cal = cal.sort_values(["y", "m", "d", "greg_dt"]).reset_index(drop=True)

    cal["Hijri_Date"] = cal["Hijri_Date"].astype(str).str.strip()
    cal = cal.drop_duplicates(subset=["Hijri_Date"], keep="first").reset_index(drop=True)

    if cache_csv:
        cal.to_csv(cache_name, index=False)

    return cal


cal_1445 = fetch_hijri_calendar_year(1445, cache_csv=True)
cal_1446 = fetch_hijri_calendar_year(1446, cache_csv=True)

cal = pd.concat([cal_1445, cal_1446], ignore_index=True)
cal = cal.sort_values(["y", "m", "d", "greg_dt"]).reset_index(drop=True)

cal["Hijri_Date"] = cal["Hijri_Date"].astype(str).str.strip()
cal = cal.drop_duplicates(subset=["Hijri_Date"], keep="first").reset_index(drop=True)

DIM = cal.groupby(["y", "m"]).size().to_dict()


def days_in_month(year, month):
    return int(DIM.get((year, month), 30))


g2h_1445 = dict(zip(pd.to_datetime(cal_1445["greg_dt"]).dt.date, cal_1445["Hijri_Date"]))
g2h_1446 = dict(zip(pd.to_datetime(cal_1446["greg_dt"]).dt.date, cal_1446["Hijri_Date"]))


# ============================================================
# (4) WEATHER
# ============================================================

def read_legacy_xls(path):
    try:
        return pd.read_excel(path, header=None, engine="xlrd")
    except Exception:
        pass

    if shutil.which("libreoffice"):
        with tempfile.TemporaryDirectory() as tmpdir:
            cmd = [
                "libreoffice",
                "--headless",
                "--convert-to",
                "csv",
                "--outdir",
                tmpdir,
                path
            ]

            subprocess.run(cmd, capture_output=True, text=True)

            csv_files = [f for f in os.listdir(tmpdir) if f.lower().endswith(".csv")]

            if csv_files:
                csv_path = os.path.join(tmpdir, csv_files[0])
                return pd.read_csv(csv_path, header=None)

    raise RuntimeError("تعذر قراءة ملف الطقس")


def load_makkah_temperature_proxy(climate_path):
    raw = read_legacy_xls(climate_path)

    makkah_idx = None

    for i in raw.index:
        row_vals = [str(v).strip() for v in raw.loc[i].tolist()]

        if any("مكة" in v for v in row_vals):
            makkah_idx = i
            break

    if makkah_idx is None:
        raise ValueError("لم يتم العثور على صف مكة في ملف الطقس")

    row = raw.loc[makkah_idx].tolist()

    numeric_vals = []

    for v in row:
        val = to_float(v)

        if pd.notna(val):
            numeric_vals.append(val)

    if len(numeric_vals) < 12:
        raise ValueError("لم أستطع استخراج 12 قيمة شهرية من ملف الطقس")

    greg_temp = {i + 1: numeric_vals[i] for i in range(12)}

    hijri_to_greg_proxy = {
        1: 7,
        2: 8,
        3: 9,
        4: 10,
        5: 11,
        6: 12,
        7: 1,
        8: 2,
        9: 3,
        10: 4,
        11: 5,
        12: 6
    }

    temp_map = {}

    for yy in [1445, 1446]:
        for h_month, g_month in hijri_to_greg_proxy.items():
            temp_map[(yy, h_month)] = greg_temp.get(g_month, np.nan)

    return temp_map


temp_map = load_makkah_temperature_proxy(CLIMATE_PATH)


def weather_weight_value(temp):
    if pd.isna(temp):
        return 1.00

    if temp <= 24:
        return 1.25
    if temp <= 30:
        return 1.15
    if temp <= 35:
        return 1.07

    return 1.00


# ============================================================
# (5) MONTHLY UMRAH
# ============================================================

def monthly_base_any_umrah_file(file_path, target_year=None):
    df = clean_cols(pd.read_excel(file_path).copy())

    col_year = pick_col(
        df,
        [
            "السنة",
            "السنة بالهجري",
            "HijriYear",
            "year",
            "السنة الهجرية"
        ]
    )

    col_month = pick_col(
        df,
        [
            "الشهر",
            "HijriMonth",
            "month"
        ]
    )

    col_count = pick_col(
        df,
        [
            "عدد تصاريح العمرة",
            "عدد التصاريح",
            "عدد تصاريح",
            "count",
            "عدد"
        ]
    )

    if col_count is None:
        count_candidates = [c for c in df.columns if "عدد" in str(c)]
        col_count = count_candidates[0] if len(count_candidates) > 0 else None

    if col_month is None or col_count is None:
        raise ValueError(f"لم أتمكن من تحديد أعمدة الشهر/العدد في الملف: {file_path}")

    if col_year is not None and target_year is not None:
        df[col_year] = pd.to_numeric(df[col_year], errors="coerce")
        df = df[df[col_year] == target_year].copy()

    df["month_no"] = df[col_month].apply(month_no_from_any)
    df[col_count] = df[col_count].apply(to_float)

    df = df.dropna(subset=["month_no", col_count]).copy()
    df["month_no"] = df["month_no"].astype(int)

    mp = df.groupby("month_no")[col_count].sum().to_dict()
    mp = {int(k): float(v) for k, v in mp.items()}

    return mp


def monthly_user_type_total(file_path, target_year, target_month_no, user_types):
    df = clean_cols(pd.read_excel(file_path).copy())

    col_year = pick_col(
        df,
        [
            "السنة",
            "السنة بالهجري",
            "HijriYear",
            "year",
            "السنة الهجرية"
        ]
    )

    col_month = pick_col(
        df,
        [
            "الشهر",
            "HijriMonth",
            "month"
        ]
    )

    col_type = pick_col(
        df,
        [
            "نوع المستخدم",
            "User Type",
            "user_type",
            "type"
        ]
    )

    col_count = pick_col(
        df,
        [
            "عدد تصاريح العمرة",
            "عدد التصاريح",
            "عدد تصاريح",
            "count",
            "عدد"
        ]
    )

    if col_year is None:
        raise ValueError("لم يتم العثور على عمود السنة في ملف نوع المستخدم")

    if col_month is None:
        raise ValueError("لم يتم العثور على عمود الشهر في ملف نوع المستخدم")

    if col_type is None:
        raise ValueError("لم يتم العثور على عمود نوع المستخدم في ملف نوع المستخدم")

    if col_count is None:
        raise ValueError("لم يتم العثور على عمود عدد تصاريح العمرة في ملف نوع المستخدم")

    df[col_year] = pd.to_numeric(df[col_year], errors="coerce")
    df["month_no"] = df[col_month].apply(month_no_from_any)
    df["user_type_clean"] = df[col_type].apply(norm_ar)
    df["count_clean"] = df[col_count].apply(to_float)

    types_clean = [norm_ar(x) for x in user_types]

    mask = (
        (df[col_year] == target_year) &
        (df["month_no"] == target_month_no) &
        (df["user_type_clean"].isin(types_clean))
    )

    sub = df.loc[mask].copy()

    if sub.empty:
        raise ValueError("لم يتم العثور على بيانات الخليجيين والمواطنين في ذو القعدة 1445")

    total = float(sub["count_clean"].sum())

    print("======================================")
    print("Dhul Qadah 1446 Replacement Source")
    print("======================================")
    print("Source Year:", target_year)
    print("Source Month:", CANON_MONTHS[target_month_no])
    print("User Types Used:", user_types)
    print("Total Used:", int(round(total)))
    print("======================================")

    return total


m1445 = monthly_base_any_umrah_file(PATH_1445_UMRAH_MONTHLY, target_year=1445)
m1446 = monthly_base_any_umrah_file(PATH_1446_UMRAH_MONTHLY, target_year=1446)

# Special fix:
# Since Dhul Qadah 1446 has no data, use Dhul Qadah 1445 GCC + Citizen only
dhul_qadah_1446_replacement_total = monthly_user_type_total(
    file_path=PATH_1445_UMRAH_MONTHLY,
    target_year=1445,
    target_month_no=11,
    user_types=["خليجي", "مواطن"]
)

m1446[11] = dhul_qadah_1446_replacement_total

monthly_map = {
    1445: m1445,
    1446: m1446
}


# ============================================================
# (6) RAMADAN DAILY MAPS
# ============================================================

def ramadan_daily_map(file_path, hijri_year):
    df = clean_cols(pd.read_excel(file_path).copy())

    col_year = pick_col(
        df,
        [
            "السنة",
            "السنة بالميلادي",
            "HijriYear",
            "year"
        ]
    )

    col_greg = pick_col(
        df,
        [
            "اليوم بالميلادي",
            "اليوم ميلادي",
            "Date_Gregorian",
            "التاريخ الميلادي",
            "GregorianDate"
        ]
    )

    col_perm = pick_col(
        df,
        [
            "عدد تصاريح رمضان",
            "عدد تصاريح العمرة",
            "عدد التصاريح",
            "عدد تصاريح",
            "permits",
            "count",
            "عدد"
        ]
    )

    if col_greg is None or col_perm is None:
        return {}

    if col_year is not None:
        df[col_year] = pd.to_numeric(df[col_year], errors="coerce")
        unique_years = set(df[col_year].dropna().astype(int).unique().tolist())

        if len(unique_years) > 0 and min(unique_years) >= 1900:
            gregorian_year = hijri_year + 579

            if gregorian_year in unique_years:
                df = df[df[col_year] == gregorian_year].copy()
        else:
            if hijri_year in unique_years:
                df = df[df[col_year] == hijri_year].copy()

    df["greg_day"] = pd.to_datetime(df[col_greg], errors="coerce").dt.date
    df["permits"] = df[col_perm].apply(to_float)

    df = df.dropna(subset=["greg_day", "permits"]).copy()

    return dict(zip(df["greg_day"], df["permits"]))


ram_map = {
    1445: ramadan_daily_map(PATH_1445_RAMADAN_DAILY, 1445),
    1446: ramadan_daily_map(PATH_1446_RAMADAN_DAILY, 1446),
}


# ============================================================
# (7) HAJJ FEATURES
# ============================================================

def hajj_raw_abroad_1445():
    df = clean_cols(pd.read_excel(PATH_1445_PILGRIMS_DAILY).copy())

    col_year = pick_col(df, ["السنة", "HijriYear", "year"])
    col_day = pick_col(df, ["اليوم", "التاريخ", "Date", "date"])
    col_male = pick_col(df, ["عدد الذكور", "الذكور", "Male"])
    col_fem = pick_col(df, ["عدد الاناث", "الاناث", "Female"])

    df[col_year] = pd.to_numeric(df[col_year], errors="coerce")
    df = df[df[col_year] == 1445].copy()

    df["greg_dt"] = pd.to_datetime(df[col_day], errors="coerce").dt.date
    df = df.dropna(subset=["greg_dt"]).copy()

    def safe_num(s):
        s = s.replace("-", np.nan) if hasattr(s, "replace") else s
        return pd.to_numeric(s, errors="coerce").fillna(0)

    male = safe_num(df[col_male]) if col_male else 0
    fem = safe_num(df[col_fem]) if col_fem else 0

    df["total"] = male + fem

    df["Hijri_Date"] = df["greg_dt"].map(g2h_1445)
    df = df.dropna(subset=["Hijri_Date"]).copy()

    df["m"] = df["Hijri_Date"].str.split("-").str[1].astype(int)
    df = df[df["m"].isin([11, 12])].copy()

    return df.groupby("Hijri_Date")["total"].sum().to_dict()


def hajj_raw_abroad_1446():
    df = clean_cols(pd.read_excel(PATH_1446_PILGRIMS_ENTER).copy())

    col_date = pick_col(df, ["تاريخ الدخول", "التاريخ", "HijriDate"])
    col_cnt = pick_col(df, ["عدد الحجاج", "count", "عدد", "العدد"])

    y, m, d = parse_hijri_ymd_yyyy_mm_dd(df[col_date])

    df["y"] = y
    df["m"] = m
    df["d"] = d
    df["cnt"] = df[col_cnt].apply(to_float)

    df = df.dropna(subset=["y", "m", "d", "cnt"]).copy()

    df["y"] = df["y"].astype(int)
    df["m"] = df["m"].astype(int)
    df["d"] = df["d"].astype(int)

    df = df[(df["y"] == 1446) & (df["m"].isin([11, 12]))].copy()

    df["Hijri_Date"] = df.apply(lambda r: HK(r["d"], r["m"], r["y"]), axis=1)

    return df.groupby("Hijri_Date")["cnt"].sum().to_dict()


def domestic_total(file_path):
    df = clean_cols(pd.read_excel(file_path).copy())

    col_cnt = pick_col(
        df,
        [
            "عدد حجاج الداخل",
            "عدد الحجاج",
            "count",
            "عدد",
            "العدد"
        ]
    )

    if col_cnt is not None:
        return float(df[col_cnt].apply(to_float).fillna(0).sum())

    num = df.select_dtypes(include=[np.number])

    return float(num.sum().sum()) if num.shape[1] else 0.0


dom_total_1445 = domestic_total(PATH_1445_DOMESTIC)
dom_total_1446 = domestic_total(PATH_1446_DOMESTIC)

DOM_W = {
    5: 1.00,
    6: 1.10,
    7: 1.20,
    8: 1.35
}


def domestic_dist_map(year, total):
    days = [5, 6, 7, 8]

    w = np.array([DOM_W[d] for d in days], dtype=float)
    w = w / w.sum()

    return {
        HK(d, 12, year): float(total) * w[i]
        for i, d in enumerate(days)
    }


dom_dist = {
    1445: domestic_dist_map(1445, dom_total_1445),
    1446: domestic_dist_map(1446, dom_total_1446),
}

hajj_raw = {
    1445: hajj_raw_abroad_1445(),
    1446: hajj_raw_abroad_1446(),
}


def build_hajj(year):
    raw_map = hajj_raw.get(year, {})
    feat = {}

    m11_days = days_in_month(year, 11)

    for d in range(1, m11_days + 1):
        k = HK(d, 11, year)
        feat[k] = float(raw_map.get(k, 0.0))

    dh_days = days_in_month(year, 12)

    for d in range(1, dh_days + 1):
        k = HK(d, 12, year)
        feat[k] = float(raw_map.get(k, 0.0))

    for k, v in dom_dist.get(year, {}).items():
        feat[k] = feat.get(k, 0.0) + float(v)

    return feat


def build_tawaf_ifadah(year):
    raw_map = hajj_raw.get(year, {})

    dh_days = days_in_month(year, 12)

    total_dh_abroad = 0.0

    for d in range(1, dh_days + 1):
        total_dh_abroad += float(raw_map.get(HK(d, 12, year), 0.0))

    total_dh_domestic = float(sum(dom_dist.get(year, {}).values()))
    total_dh = total_dh_abroad + total_dh_domestic

    dist_days = list(range(10, dh_days + 1))

    if len(dist_days) == 0:
        return {}

    weights = np.linspace(3.0, 0.5, len(dist_days))
    weights = weights / weights.sum()

    tawaf = {}

    for i, d in enumerate(dist_days):
        tawaf[HK(d, 12, year)] = float(total_dh) * float(weights[i])

    return tawaf


hajj = {
    1445: build_hajj(1445),
    1446: build_hajj(1446)
}

tawaf_ifadah = {
    1445: build_tawaf_ifadah(1445),
    1446: build_tawaf_ifadah(1446)
}


# ============================================================
# (8) UMRAH WEIGHTS
# ============================================================

THU_W = 1.25
FRI_W = 1.35
SAT_W = 1.15
WK_W = 1.00

DECEMBER_W = 1.10


# ============================================================
# (9) BUILD FINAL DATASET FROM SCRATCH
# ============================================================

rows = []

for year in [1445, 1446]:

    cal_y = cal[cal["y"] == year].copy()

    for _, r in cal_y.iterrows():

        m = int(r["m"])
        hijri_date = str(r["Hijri_Date"]).strip()
        month_name = r["Hijri_Month_Name"]

        greg_day = r["greg_dt"]
        day_en = str(r["Day_EN"]).strip()
        day_ar = str(r["Weekday_AR"]).strip()

        dim = days_in_month(year, m)

        monthly_total = monthly_map[year].get(m, np.nan)

        base_monthly = (
            monthly_total / dim
            if dim and pd.notna(monthly_total)
            else np.nan
        )

        avg_temp = float(temp_map.get((year, m), np.nan))
        wweather = weather_weight_value(avg_temp)

        if m == 9 and len(ram_map.get(year, {})) > 0:
            if pd.notna(base_monthly):
                base = float(ram_map[year].get(greg_day, base_monthly))
            else:
                base = float(ram_map[year].get(greg_day, np.nan))
        else:
            base = base_monthly

        if pd.isna(base):
            umrah = np.nan
        else:
            if day_en == "Thursday":
                wday = THU_W
            elif day_en == "Friday":
                wday = FRI_W
            elif day_en == "Saturday":
                wday = SAT_W
            else:
                wday = WK_W

            wseason = DECEMBER_W if pd.to_datetime(greg_day).month == 12 else 1.00

            umrah = round(base * wday * wseason * wweather)

        hajj_value = float(hajj[year].get(hijri_date, 0.0)) if m in [11, 12] else 0.0
        tawaf_value = float(tawaf_ifadah[year].get(hijri_date, 0.0)) if m == 12 else 0.0

        rows.append({
            "Hijri_Year": year,
            "Hijri_Date": hijri_date,
            "Hijri_Month": month_name,
            "Gregorian_Date": str(greg_day),
            "Weekday_AR": day_ar,
            "AvgTemp_C": np.nan if pd.isna(avg_temp) else round(avg_temp, 3),
            "المعتمرين": np.nan if pd.isna(umrah) else int(round(umrah)),
            "Hajj": int(round(hajj_value)),
            "Tawaf_Ifadah": int(round(tawaf_value))
        })


df = pd.DataFrame(rows)


# ============================================================
# (9.1) FORCE NON-RAMADAN MONTHS TO MATCH RAW TOTALS
# ============================================================

def force_month_exact_total(df, year, month_no, target_total):
    month_name = CANON_MONTHS[month_no]

    mask = (
        (df["Hijri_Year"] == year) &
        (df["Hijri_Month"] == month_name)
    )

    sub = df.loc[mask].copy()

    if sub.empty:
        return df

    current_values = pd.to_numeric(
        sub["المعتمرين"],
        errors="coerce"
    ).fillna(0).values

    current_sum = current_values.sum()
    n = len(sub)

    target_total = int(round(target_total))

    if current_sum <= 0:
        base = int(target_total // n)
        remainder = int(target_total - base * n)

        new_values = np.array([base] * n, dtype=int)

        for i in range(remainder):
            new_values[i % n] += 1

    else:
        scaled = current_values * (target_total / current_sum)
        floored = np.floor(scaled).astype(int)

        remainder = int(round(target_total - floored.sum()))

        frac = scaled - floored
        order = np.argsort(-frac)

        new_values = floored.copy()

        for i in range(remainder):
            new_values[order[i % len(order)]] += 1

    df.loc[mask, "المعتمرين"] = new_values

    return df


for year in [1445, 1446]:
    for month_no in range(1, 13):

        if month_no == 9:
            continue

        if month_no in monthly_map[year]:
            target_total = int(round(monthly_map[year][month_no]))
        else:
            continue

        df = force_month_exact_total(
            df=df,
            year=year,
            month_no=month_no,
            target_total=target_total
        )


# ============================================================
# (10) CROWDING LEVELS
# ============================================================

df["Crowding_Level"] = pd.Series([None] * len(df), dtype="object")
df["Crowding_Level_Num"] = np.nan

for year in [1445, 1446]:

    sub = df[df["Hijri_Year"] == year].copy()
    v = pd.to_numeric(sub["المعتمرين"], errors="coerce").dropna()

    if len(v) == 0:
        continue

    q1, q2, q3 = v.quantile([0.25, 0.50, 0.75])

    def cl(x):
        if pd.isna(x):
            return None
        if x <= q1:
            return "Low"
        if x <= q2:
            return "Medium"
        if x <= q3:
            return "High"
        return "Critical"

    mask = df["Hijri_Year"] == year

    df.loc[mask, "Crowding_Level"] = df.loc[mask, "المعتمرين"].apply(cl)

    df.loc[mask, "Crowding_Level_Num"] = df.loc[mask, "Crowding_Level"].map({
        "Low": 1,
        "Medium": 2,
        "High": 3,
        "Critical": 4
    })


# ============================================================
# (11) FINAL CLEANING AND ORDER
# ============================================================

for c in REQUIRED_COLS:
    if c not in df.columns:
        df[c] = np.nan

df = df[REQUIRED_COLS]

df["المعتمرين"] = pd.to_numeric(df["المعتمرين"], errors="coerce").round().astype("Int64")
df["Hajj"] = pd.to_numeric(df["Hajj"], errors="coerce").fillna(0).round().astype(int)
df["Tawaf_Ifadah"] = pd.to_numeric(df["Tawaf_Ifadah"], errors="coerce").fillna(0).round().astype(int)
df["Crowding_Level_Num"] = pd.to_numeric(df["Crowding_Level_Num"], errors="coerce").astype("Int64")


# ============================================================
# (12) SAVE OUTPUT
# ============================================================

df.to_excel(OUT_FILE, index=False)

wb = load_workbook(OUT_FILE)
ws = wb.active

headers = [cell.value for cell in ws[1]]

if "Hijri_Date" in headers:
    hcol = headers.index("Hijri_Date") + 1

    for row in range(2, ws.max_row + 1):
        ws.cell(row=row, column=hcol).number_format = "@"

if "Gregorian_Date" in headers:
    gcol = headers.index("Gregorian_Date") + 1

    for row in range(2, ws.max_row + 1):
        ws.cell(row=row, column=gcol).number_format = "@"

wb.save(OUT_FILE)


# ============================================================
# (13) FINAL CHECK
# ============================================================

print("======================================")
print("DONE")
print("======================================")
print("Saved file:", OUT_FILE)

print("======================================")
print("CHECK DHUL QADAH 1446")
print("======================================")

check_df = df[
    (df["Hijri_Year"] == 1446) &
    (df["Hijri_Month"] == "ذو القعدة")
].copy()

print(check_df[
    [
        "Hijri_Year",
        "Hijri_Date",
        "Hijri_Month",
        "Gregorian_Date",
        "Weekday_AR",
        "AvgTemp_C",
        "المعتمرين",
        "Hajj",
        "Tawaf_Ifadah",
        "Crowding_Level",
        "Crowding_Level_Num"
    ]
])

print("--------------------------------------")
print("Dhul Qadah 1446 Total:", int(check_df["المعتمرين"].sum()))
print("Expected Total From 1445 GCC + Citizen:", int(round(dhul_qadah_1446_replacement_total)))

print("======================================")
print("FINAL COLUMNS")
print("======================================")
print(df.columns.tolist())

Dhul Qadah 1446 Replacement Source
Source Year: 1445
Source Month: ذو القعدة
User Types Used: ['خليجي', 'مواطن']
Total Used: 123427
DONE
Saved file: /content/Umrah_Crowd_Forecasting_Dataset_Updated.xlsx
CHECK DHUL QADAH 1446
     Hijri_Year  Hijri_Date Hijri_Month Gregorian_Date Weekday_AR  AvgTemp_C  \
650        1446  01-11-1446   ذو القعدة     2025-04-29   الثلاثاء       43.0   
651        1446  02-11-1446   ذو القعدة     2025-04-30   الأربعاء       43.0   
652        1446  03-11-1446   ذو القعدة     2025-05-01     الخميس       43.0   
653        1446  04-11-1446   ذو القعدة     2025-05-02     الجمعة       43.0   
654        1446  05-11-1446   ذو القعدة     2025-05-03      السبت       43.0   
655        1446  06-11-1446   ذو القعدة     2025-05-04      الأحد       43.0   
656        1446  07-11-1446   ذو القعدة     2025-05-05    الاثنين       43.0   
657        1446  08-11-1446   ذو القعدة     2025-05-06   الثلاثاء       43.0   
658        1446  09-11-1446   ذو القعدة     2025-05-07 

In [2]:
from google.colab import files

# اسم الملف القديم المطلوب للتحميل
OLD_NAME_FILE = "/content/Intelligent System for Predicting Visitor Crowd Levels and Optimal Visiting Times at Al-Masjid Al-Haram.xlsx"

# حفظ نفس الداتا الجديدة لكن باسم الملف القديم
df.to_excel(OLD_NAME_FILE, index=False)

# تحميل الملف
files.download(OLD_NAME_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        print(os.path.join(root, file))

/content/Number of pilgrims entering by age group for 1446.xlsx
/content/hijri_calendar_1445.csv
/content/Umrah permits by day for the month of Ramadan 1446.xlsx
/content/Umrah permits for the year 1446 by month and type.xlsx
/content/Number of domestic pilgrims by age group for 1446.xlsx
/content/Number of domestic pilgrims in 1445 by age.xlsx
/content/Number of Umrah permits in Ramadan.xlsx
/content/Umrah_Crowd_Forecasting_Dataset_Updated.xlsx
/content/Intelligent System for Predicting Visitor Crowd Levels and Optimal Visiting Times at Al-Masjid Al-Haram.xlsx
/content/Number of pilgrims from abroad by gender per day.xlsx
/content/hijri_calendar_1446.csv
/content/Average Maximum Temperature, By Month and Meteorological   Station2001 A.D. ( In Centigrade Degrees )_2001.xls
/content/Number of Umrah permits by user type.xlsx
/content/.config/gce
/content/.config/default_configs.db
/content/.config/config_sentinel
/content/.config/.last_survey_prompt.yaml
/content/.config/active_config
/c